# AIEngineer 7주차 — 유형준 강사님 풀이 (Blind, Part 1 코드 구현)
- 문제지만 보고 작성. 정답지 미참조.
- Part 2 (문제 8~10)는 MCP 서버/Inspector/curl/pytest 캡처 기반 → 코드 외 증빙 중심 과제라 별도 패키지 필요. 본 노트북은 Part 1 (1~7번) 코드 구현만 포함.

In [ ]:
!pip install -q pyyaml

In [ ]:
import json, sqlite3, hashlib, re, time, asyncio, inspect, csv, io, os, tempfile, typing, yaml
from pathlib import Path
from dataclasses import dataclass, field, asdict
from enum import Enum
from datetime import datetime, timezone

## 문제 1. 타입 힌트 → JSON Schema

In [ ]:
def python_type_to_json_schema(py_type) -> str:
    mapping = {str: 'string', int: 'integer', float: 'number', bool: 'boolean'}
    return mapping.get(py_type, 'string')


def extract_tool_schema(func) -> dict:
    sig = inspect.signature(func)
    properties = {}
    required = []
    for name, param in sig.parameters.items():
        if name == 'ctx':
            continue
        py_type = param.annotation if param.annotation is not inspect.Parameter.empty else str
        properties[name] = {'type': python_type_to_json_schema(py_type)}
        if param.default is inspect.Parameter.empty:
            required.append(name)
    return {
        'name': func.__name__,
        'description': (func.__doc__ or '').strip(),
        'inputSchema': {
            'type': 'object',
            'properties': properties,
            'required': required,
        },
    }

In [ ]:
def lookup_inventory(query: str, limit: int = 10, ctx=None) -> str:
    """Search inventory items by keyword."""
    pass

schema = extract_tool_schema(lookup_inventory)
print(json.dumps(schema, indent=2))

## 문제 2. CSV → SQLite 재고 조회

In [ ]:
INVENTORY_CSV = """item_id,name,category,quantity,location,status,last_updated
INV-001,MacBook Pro 14 M3 Pro,Electronics,25,Warehouse A,in_stock,2026-01-15
INV-002,Dell Latitude 5540,Electronics,30,Warehouse B,in_stock,2026-01-20
INV-003,Lenovo ThinkPad X1 Carbon,Electronics,15,Warehouse A,in_stock,2026-01-18
INV-004,Dell UltraSharp U2723QE Monitor,Electronics,40,Warehouse C,in_stock,2026-01-10
INV-005,Logitech MX Keys Keyboard,Peripherals,100,Warehouse B,in_stock,2026-01-22
INV-006,Apple Magic Mouse,Peripherals,60,Warehouse A,in_stock,2026-01-12
INV-007,Jabra Evolve2 75 Headset,Peripherals,45,Warehouse C,in_stock,2026-01-25
INV-008,CalDigit TS4 Docking Station,Peripherals,20,Warehouse A,low_stock,2026-01-08
INV-009,iPad Air M2,Electronics,10,Warehouse B,low_stock,2026-01-14
INV-010,HP LaserJet Pro M404dn,Office,8,Warehouse C,in_stock,2026-01-30
INV-011,Office Chair ErgoMax Pro,Furniture,50,Warehouse A,in_stock,2026-02-01
INV-012,Standing Desk VariDesk Pro,Furniture,12,Warehouse B,low_stock,2026-01-28"""


def init_inventory_db(csv_data: str) -> sqlite3.Connection:
    db = sqlite3.connect(':memory:')
    db.row_factory = sqlite3.Row
    db.execute(
        '''CREATE TABLE inventory (
            item_id TEXT PRIMARY KEY,
            name TEXT,
            category TEXT,
            quantity INTEGER,
            location TEXT,
            status TEXT,
            last_updated TEXT
        )'''
    )
    reader = csv.DictReader(io.StringIO(csv_data))
    rows = [
        (r['item_id'], r['name'], r['category'], int(r['quantity']),
         r['location'], r['status'], r['last_updated'])
        for r in reader
    ]
    db.executemany('INSERT INTO inventory VALUES (?, ?, ?, ?, ?, ?, ?)', rows)
    db.commit()
    return db


def search_inventory(db: sqlite3.Connection, query: str) -> list[dict]:
    sql = (
        'SELECT * FROM inventory '
        'WHERE LOWER(name) LIKE ? OR LOWER(category) LIKE ? '
        'LIMIT 10'
    )
    like = f'%{query.lower()}%'
    cur = db.execute(sql, (like, like))
    return [dict(row) for row in cur.fetchall()]

In [ ]:
db = init_inventory_db(INVENTORY_CSV)
results = search_inventory(db, 'dell')
print(f"'dell' 검색 결과: {len(results)}건")
for r in results:
    print(f"  - {r['item_id']}: {r['name']} ({r['category']})")

results2 = search_inventory(db, 'peripherals')
print(f"\n'peripherals' 검색 결과: {len(results2)}건")

results3 = search_inventory(db, 'xyz_not_exist')
print(f"\n'xyz_not_exist' 검색 결과: {len(results3)}건")

## 문제 3. ToolError 패턴

In [ ]:
class ErrorCode(Enum):
    NOT_FOUND = 'NOT_FOUND'
    INVALID_ARGUMENT = 'INVALID_ARGUMENT'
    PERMISSION_DENIED = 'PERMISSION_DENIED'
    INTERNAL_ERROR = 'INTERNAL_ERROR'
    CONFLICT = 'CONFLICT'


class ToolError(Exception):
    def __init__(self, code: ErrorCode, message: str):
        super().__init__(message)
        self.code = code
        self.message = message

    def __str__(self):
        return f'[{self.code.value}] {self.message}'


def safe_search(db: sqlite3.Connection, query: str) -> list[dict]:
    if not isinstance(query, str) or not query.strip():
        raise ToolError(ErrorCode.INVALID_ARGUMENT, '검색어를 입력하세요')
    try:
        return search_inventory(db, query.strip())
    except ToolError:
        raise
    except Exception as e:
        raise ToolError(ErrorCode.INTERNAL_ERROR, f'검색 중 오류: {e}')

In [ ]:
results = safe_search(db, 'macbook')
print(f"정상 검색: {len(results)}건 - {results[0]['name']}")

empty = safe_search(db, 'xyz')
print(f'빈 결과: {empty} (type: {type(empty).__name__})')

try:
    safe_search(db, '')
except ToolError as e:
    print(f'빈 입력 에러: {e}')

try:
    safe_search(db, '   ')
except ToolError as e:
    print(f'공백 입력 에러: {e}')

## 문제 4. YAML 파싱 + 관련도 랭킹

In [ ]:
POLICY_DOC_1 = """---
title: VPN 설정 가이드
tags: [vpn, 네트워크, 보안, 원격근무]
last_updated: 2026-01-15
---
# VPN 설정 가이드

## 개요
사내 VPN은 WireGuard를 기본으로 사용합니다.

## 설치 방법
macOS에서는 Homebrew로 설치합니다.
VPN 연결이 필요한 경우 IT팀에 문의하세요.
VPN 설정 후 반드시 연결 테스트를 수행하세요."""

POLICY_DOC_2 = """---
title: 보안 정책
tags: [보안, 비밀번호, 인증, 컴플라이언스]
last_updated: 2026-02-01
---
# 보안 정책

## 비밀번호 규칙
비밀번호는 14자 이상이어야 합니다.

## MFA 필수
모든 직원은 MFA를 활성화해야 합니다.
VPN 접속 시에도 MFA 인증이 필요합니다."""

POLICY_DOC_3 = """---
title: 원격근무 정책
tags: [원격근무, 재택, 하이브리드]
last_updated: 2026-01-20
---
# 원격근무 정책

## 핵심 근무 시간
10:00~16:00 사이에는 반드시 연락 가능해야 합니다.

## 장비 지원
노트북, 모니터 1대, 키보드, 마우스, 헤드셋을 지원합니다."""


def parse_frontmatter(content: str) -> tuple[dict, str]:
    if not content.startswith('---'):
        return {}, content
    parts = content.split('---', 2)
    # parts = ['', yaml_str, body]
    if len(parts) < 3:
        return {}, content
    meta = yaml.safe_load(parts[1]) or {}
    body = parts[2].lstrip('\n')
    return meta, body


def calculate_relevance(query: str, title: str, tags: list[str], body: str) -> float:
    q = query.lower()
    score = 0.0
    if q in (title or '').lower():
        score += 3.0
    if any(q in (t or '').lower() for t in (tags or [])):
        score += 2.0
    occurrences = (body or '').lower().count(q)
    score += min(occurrences * 0.5, 3.0)
    return score

In [ ]:
meta1, body1 = parse_frontmatter(POLICY_DOC_1)
print(f"제목: {meta1['title']}")
print(f"태그: {meta1['tags']}")
print(f'본문 길이: {len(body1)}자')

docs = [POLICY_DOC_1, POLICY_DOC_2, POLICY_DOC_3]
print("\n=== 'vpn' 검색 관련도 ===")
for i, doc in enumerate(docs, 1):
    meta, body = parse_frontmatter(doc)
    score = calculate_relevance('vpn', meta['title'], meta['tags'], body)
    print(f"문서{i} ({meta['title']}): {score}점")

## 문제 5. Confirm Gate + 멱등성

In [ ]:
def create_ticket(title, body, priority, confirm, tickets_store):
    if not confirm:
        return f'[미리보기] 제목: {title} | 우선순위: {priority} — confirm=True로 생성하세요.'
    ticket_id = f'TKT-{len(tickets_store) + 1:03d}'
    ticket = {
        'ticket_id': ticket_id,
        'title': title,
        'body': body,
        'priority': priority,
        'status': 'open',
        'created_at': datetime.now(timezone.utc).isoformat(),
    }
    tickets_store.append(ticket)
    return ticket


def create_ticket_idempotent(title, body, priority, confirm, tickets_store):
    if not confirm:
        return f'[미리보기] 제목: {title} | 우선순위: {priority} — confirm=True로 생성하세요.'

    key = hashlib.sha256((title + body).encode()).hexdigest()[:16]

    for existing in tickets_store:
        if existing.get('idempotency_key') == key:
            return existing

    ticket_id = f'TKT-{len(tickets_store) + 1:03d}'
    ticket = {
        'ticket_id': ticket_id,
        'title': title,
        'body': body,
        'priority': priority,
        'status': 'open',
        'created_at': datetime.now(timezone.utc).isoformat(),
        'idempotency_key': key,
    }
    tickets_store.append(ticket)
    return ticket

In [ ]:
store = []

preview = create_ticket_idempotent('프린터 고장', '2층 프린터가 작동하지 않습니다.', 'medium', confirm=False, tickets_store=store)
print(f'미리보기: {preview}')
print(f'저장소 변경 없음: {len(store)}건')

ticket1 = create_ticket_idempotent('프린터 고장', '2층 프린터가 작동하지 않습니다.', 'medium', confirm=True, tickets_store=store)
print(f"\n생성된 티켓: {ticket1['ticket_id']} - {ticket1['title']}")
print(f'저장소: {len(store)}건')

ticket2 = create_ticket_idempotent('프린터 고장', '2층 프린터가 작동하지 않습니다.', 'medium', confirm=True, tickets_store=store)
print(f"\n중복 시도 결과: {ticket2['ticket_id']} (기존 티켓 반환)")
print(f'저장소 변경 없음: {len(store)}건')

ticket3 = create_ticket_idempotent('모니터 깜빡임', '3층 모니터가 깜빡입니다.', 'high', confirm=True, tickets_store=store)
print(f"\n새 티켓: {ticket3['ticket_id']} - {ticket3['title']}")
print(f'저장소: {len(store)}건')

## 문제 6. 입력 검증 (sanitize + validate_doc_id)

In [ ]:
CONTROL_CHARS = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')

DANGEROUS_PATTERNS = [
    re.compile(r"['\\\";]\s*(?:DROP|DELETE|UPDATE|INSERT|ALTER|EXEC)", re.IGNORECASE),
    re.compile(r'<\s*script', re.IGNORECASE),
    re.compile(r'\$(?:gt|lt|ne|eq|regex|where|or|and)\b'),
    re.compile(r'__proto__|constructor\s*\['),
]


def sanitize_string(value: str, max_length: int = 500) -> str:
    if not isinstance(value, str):
        raise ValueError('문자열이 아닙니다')
    v = value.strip()
    v = CONTROL_CHARS.sub('', v)
    if len(v) > max_length:
        v = v[:max_length]
    for pat in DANGEROUS_PATTERNS:
        if pat.search(v):
            raise ValueError('위험한 패턴이 감지되었습니다')
    return v


DOC_ID_PATTERN = re.compile(r'^[a-z0-9]([a-z0-9\-]{0,48}[a-z0-9])?$')


def validate_doc_id(doc_id: str) -> str:
    if not isinstance(doc_id, str):
        raise ValueError('문자열이 아닙니다')
    v = doc_id.strip()
    if not v:
        raise ValueError('빈 doc_id')
    if any(ch in v for ch in ('..', '/', '\\', '\x00')):
        raise ValueError('허용되지 않는 문자')
    if not DOC_ID_PATTERN.match(v):
        raise ValueError('doc_id 형식 오류')
    return v

In [ ]:
print('=== sanitize_string ===')
print(f"정상: '{sanitize_string('  hello world  ')}'")
print(f"길이제한: '{sanitize_string('a' * 600, max_length=10)}'")

for attack in ["'; DROP TABLE users; --", '<script>alert(1)</script>', '$gt value']:
    try:
        sanitize_string(attack)
        print(f'통과 (오류!): {attack}')
    except ValueError:
        print(f'차단됨: {attack[:30]}')

print('\n=== validate_doc_id ===')
for valid_id in ['vpn-setup', 'remote-work', 'a', 'abc123']:
    print(f'통과: {validate_doc_id(valid_id)}')

for bad_id in ['../../../etc/passwd', 'foo/bar', 'UPPER', '', 'a' * 60, 'hello\x00world']:
    try:
        validate_doc_id(bad_id)
        print(f'통과 (오류!): {bad_id}')
    except ValueError:
        print(f'차단됨: {repr(bad_id)[:30]}')

## 문제 7. Resource + Prompt 템플릿

In [ ]:
@dataclass
class PolicyDoc:
    doc_id: str
    title: str
    path: Path
    tags: list[str]


def build_policy_index(policies: list[PolicyDoc]) -> list[dict]:
    return [
        {'doc_id': p.doc_id, 'title': p.title, 'tags': p.tags}
        for p in policies
    ]


def get_policy_content(policies: list[PolicyDoc], doc_id: str, policy_dir: Path) -> dict:
    safe_id = validate_doc_id(doc_id)
    for p in policies:
        if p.doc_id == safe_id:
            content = p.path.read_text(encoding='utf-8')
            return {'doc_id': p.doc_id, 'title': p.title, 'content': content}
    raise ValueError(f'문서를 찾을 수 없음: {safe_id}')


def incident_report_prompt(issue: str, affected_system: str) -> str:
    return (
        'IT 인시던트 보고서를 다음 형식으로 작성하세요.\n\n'
        f'이슈: {issue}\n'
        f'영향 시스템: {affected_system}\n\n'
        '[보고서 항목]\n'
        '1. 요약: 인시던트 한 줄 요약\n'
        '2. 영향 범위: 영향을 받은 사용자/시스템/서비스 범위\n'
        '3. 재현 절차: 문제를 재현하는 단계별 절차\n'
        '4. 권장 조치: 단기 대응 및 재발 방지를 위한 권장 조치\n'
        '5. 우선순위: low/medium/high/critical 중 하나와 그 근거\n'
    )

In [ ]:
tmp = Path(tempfile.mkdtemp())
(tmp / 'vpn-setup.md').write_text(POLICY_DOC_1, encoding='utf-8')
(tmp / 'security-policy.md').write_text(POLICY_DOC_2, encoding='utf-8')

policies = [
    PolicyDoc('vpn-setup', 'VPN 설정 가이드', tmp / 'vpn-setup.md', ['vpn', '네트워크']),
    PolicyDoc('security-policy', '보안 정책', tmp / 'security-policy.md', ['보안', '인증']),
]

index = build_policy_index(policies)
print(f'인덱스: {len(index)}개')
print(json.dumps(index, ensure_ascii=False, indent=2))

detail = get_policy_content(policies, 'vpn-setup', tmp)
print(f"\n상세: {detail['doc_id']} - {detail['title']} ({len(detail['content'])}자)")

try:
    get_policy_content(policies, '../../etc/passwd', tmp)
except ValueError:
    print('Path Traversal 차단됨')

prompt = incident_report_prompt('2층 프린터 작동 중단', 'HP LaserJet Pro M404dn')
for item in ['요약', '영향 범위', '재현 절차', '권장 조치', '우선순위']:
    print(f"  {item}: {'✓' if item in prompt else '✗'}")
print(f"  이슈 포함: {'✓' if '프린터' in prompt else '✗'}")
print(f"  시스템 포함: {'✓' if 'LaserJet' in prompt else '✗'}")

## Part 2 (문제 8~10) 메모

- **문제 8**: starter-kit 기반 MCP 서버 구축 + Inspector로 `lookup_inventory` Tool, `policy://index` Resource 검증. Inspector Tool/Resource 목록 캡처, Tool 호출 JSON-RPC 메시지 캡처 필요.
- **문제 9**: Transport를 stdio → Streamable HTTP(port 8000)로 전환. `curl -X POST http://localhost:8000/mcp`로 `tools/list` 및 `tools/call` 호출 후 응답 캡처.
- **문제 10**: `test_inventory.py(3+)`, `test_validation.py(5+)`, `test_ticket.py(3+)` 최소 11개 테스트 작성, `pytest --cov=src --cov-report=term-missing -v` 커버리지 90%+ 캡처.

이 세 문항은 외부 실행/캡처가 필수인 Part 2 킬러 문항이라 본 노트북에서 재현 불가. 실제 제출은 `problem-8/answer/`, `problem-9/answer/`, `problem-10/answer/` 디렉토리에 src, png, txt 형태로 구성 필요.